In [ ]:
import json
import numpy as np
import torch

from pathlib import Path
from sentence_transformers import SentenceTransformer


# ============================================================
# Configuration
# ============================================================

# Patient chunks produced by Notebook 3
CHUNKS_PATH = Path("all_patient_chunks.jsonl")

# Precomputed patient chunk embeddings produced by Notebook 4
EMBEDDINGS_PATH = Path("patient_chunk_embeddings.npy")

# Embedding model
EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


print(f"Device: {DEVICE}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Chunks file: {CHUNKS_PATH}")
print(f"Embeddings file: {EMBEDDINGS_PATH}")

In [ ]:
# Load chunks and embeddings

# ============================================================
# Load patient chunks
# ============================================================

patient_chunks = []

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            patient_chunks.append(json.loads(line))


# ============================================================
# Load precomputed embeddings
# ============================================================

patient_embeddings = np.load(EMBEDDINGS_PATH)


# ============================================================
# Verify alignment
# ============================================================

assert len(patient_chunks) == len(patient_embeddings), (
    f"Mismatch: {len(patient_chunks)} chunks vs "
    f"{len(patient_embeddings)} embeddings"
)

print(f"Patient chunks loaded: {len(patient_chunks)}")
print(f"Embedding matrix shape: {patient_embeddings.shape}")
print("Chunk/embedding alignment verified.")

In [ ]:
# ============================================================
# Load embedding model for retrieval queries
# ============================================================

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=DEVICE
)

print("Embedding model loaded successfully.")
print(f"Embedding dimension: {embedding_model.get_embedding_dimension()}")

In [ ]:
# ============================================================
# Patient-case RAG retrieval
# ============================================================

def retrieve_patient_chunks(
    hadm_id,
    query,
    top_k=10
):
    """
    Retrieve the most relevant chunks from a single patient's
    admission using a disease-specific retrieval query.

    Parameters
    ----------
    hadm_id : int
        Hospital admission ID.

    query : str
        Disease-specific retrieval query.

    top_k : int
        Number of patient chunks to retrieve.

    Returns
    -------
    list of dict
        Retrieved chunks with similarity scores.
    """

    # 1. Find chunks belonging only to this admission
    admission_indices = [
        i
        for i, chunk in enumerate(patient_chunks)
        if str(chunk["hadm_id"]) == str(hadm_id)
    ]

    if not admission_indices:
        raise ValueError(
            f"No patient chunks found for hadm_id={hadm_id}"
        )

    # 2. Embed the disease-specific query
    query_embedding = embedding_model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # 3. Get embeddings belonging only to this admission
    admission_embeddings = patient_embeddings[admission_indices]

    # Embeddings are normalized:
    # dot product = cosine similarity
    similarities = admission_embeddings @ query_embedding

    # 4. Rank chunks
    k = min(top_k, len(admission_indices))
    top_local_indices = np.argsort(similarities)[::-1][:k]

    # 5. Return chunks + similarity scores
    results = []

    for local_idx in top_local_indices:

        global_idx = admission_indices[local_idx]

        result = patient_chunks[global_idx].copy()
        result["similarity_score"] = float(
            similarities[local_idx]
        )

        results.append(result)

    return results

In [ ]:
# ============================================================
# Test patient-case RAG
# ============================================================

# TEST_HADM_ID = 00000000

# hf_results = retrieve_patient_chunks(
#     hadm_id=TEST_HADM_ID,
#     disease_name="Heart failure",
#     specialty="Cardiology",
#     top_k=10
# )

# print(f"Retrieved {len(hf_results)} chunks")
# print(f"HADM ID: {TEST_HADM_ID}")
# print("=" * 100)

# for rank, result in enumerate(hf_results, start=1):

#     print(f"\nRANK {rank}")
#     print(f"Similarity: {result['similarity_score']:.4f}")
#     print(f"Section: {result['section']}")
#     print(f"Chunk ID: {result['chunk_id']}")
#     print("-" * 100)
#     print(result["text"])
#     print("=" * 100)

In [ ]:
# ============================================================
# Disease-specific Patient-RAG queries: Cardiology
# ============================================================

cardiology_queries = {

    "CIR017": (
        "Evidence relevant to cardiac dysrhythmias, including the patient's "
        "cardiac rhythm, rate, conduction, electrocardiographic findings, "
        "rhythm-related symptoms, cardiac history, and rhythm-related "
        "treatments or procedures."
    ),

    "CIR019": (
        "Evidence relevant to heart failure, including cardiac function, "
        "volume status, symptoms and signs of cardiac congestion, "
        "cardiopulmonary findings, cardiac investigations, and treatments "
        "related to cardiac function or volume management."
    ),

    "CIR008": (
        "Evidence relevant to hypertension with complications or secondary "
        "hypertension, including blood pressure abnormalities, possible "
        "secondary causes, associated organ dysfunction, cardiovascular or "
        "renal complications, and antihypertensive management."
    ),

    "CIR011": (
        "Evidence relevant to coronary atherosclerosis and other ischemic "
        "heart disease, including ischemic symptoms, cardiac biomarkers, "
        "electrocardiographic findings, cardiac investigations, coronary "
        "history, and treatments related to myocardial ischemia."
    ),

    "CIR007": (
        "Evidence relevant to essential hypertension, including blood pressure "
        "measurements and trends, history of hypertension, cardiovascular "
        "assessment, and antihypertensive treatment."
    )
}

In [ ]:
# ============================================================
# Compare disease-specific retrievals for Cardiology
# ============================================================

cardiology_diseases = {
    "CIR017": "Cardiac dysrhythmias",
    "CIR019": "Heart failure",
    "CIR008": "Hypertension with complications and secondary hypertension",
    "CIR011": "Coronary atherosclerosis and other heart disease",
    "CIR007": "Essential hypertension"
}

TEST_HADM_ID = 00000000
TOP_K = 10

cardiology_results = {}

for disease_code, disease_name in cardiology_diseases.items():

    results = retrieve_patient_chunks(
        hadm_id=TEST_HADM_ID,
        query=cardiology_queries[disease_code],
        top_k=TOP_K
    )

    cardiology_results[disease_code] = results

    print("\n" + "=" * 110)
    print(f"{disease_code} - {disease_name}")
    print("=" * 110)

    for rank, result in enumerate(results, start=1):
        print(
            f"{rank:>2}. "
            f"Score={result['similarity_score']:.4f} | "
            f"Section={result['section']} | "
            f"Chunk ID={result['chunk_id']}"
        )

        # Short preview to make comparison easier
        preview = " ".join(result["text"].split())
        print(f"    {preview[:250]}")

In [ ]:
# ============================================================
# Compare overlap between cardiology disease retrievals
# ============================================================

from itertools import combinations

for code_1, code_2 in combinations(cardiology_diseases.keys(), 2):

    chunks_1 = {
        result["chunk_id"]
        for result in cardiology_results[code_1]
    }

    chunks_2 = {
        result["chunk_id"]
        for result in cardiology_results[code_2]
    }

    shared_chunks = chunks_1.intersection(chunks_2)

    overlap_count = len(shared_chunks)
    overlap_percent = (overlap_count / TOP_K) * 100

    print(
        f"{code_1} vs {code_2}: "
        f"{overlap_count}/{TOP_K} shared "
        f"({overlap_percent:.0f}%)"
    )

    print(f"    Shared chunk IDs: {sorted(shared_chunks)}")

In [ ]:

# ============================================================
# Compare disease-specific retrievals for Cardiology
# ============================================================

cardiology_diseases = {
    "CIR017": "Cardiac dysrhythmias",
    "CIR019": "Heart failure",
    "CIR008": "Hypertension with complications and secondary hypertension",
    "CIR011": "Coronary atherosclerosis and other heart disease",
    "CIR007": "Essential hypertension"
}

TEST_HADM_ID = 00000000
TOP_K = 10

cardiology_results = {}

for disease_code, disease_name in cardiology_diseases.items():

    results = retrieve_patient_chunks(
        hadm_id=TEST_HADM_ID,
        query=cardiology_queries[disease_code],
        top_k=TOP_K
    )

    cardiology_results[disease_code] = results

    print("\n" + "=" * 110)
    print(f"{disease_code} - {disease_name}")
    print("=" * 110)

    for rank, result in enumerate(results, start=1):
        print(
            f"{rank:>2}. "
            f"Score={result['similarity_score']:.4f} | "
            f"Section={result['section']} | "
            f"Chunk ID={result['chunk_id']}"
        )

        # Short preview to make comparison easier
        preview = " ".join(result["text"].split())
        print(f"    {preview[:250]}")

In [ ]:

# ============================================================
# Compare disease-specific retrievals for Cardiology
# ============================================================

cardiology_diseases = {
    "CIR017": "Cardiac dysrhythmias",
    "CIR019": "Heart failure",
    "CIR008": "Hypertension with complications and secondary hypertension",
    "CIR011": "Coronary atherosclerosis and other heart disease",
    "CIR007": "Essential hypertension"
}

TEST_HADM_ID = 00000000
TOP_K = 10

cardiology_results = {}

for disease_code, disease_name in cardiology_diseases.items():

    results = retrieve_patient_chunks(
        hadm_id=TEST_HADM_ID,
        query=cardiology_queries[disease_code],
        top_k=TOP_K
    )

    cardiology_results[disease_code] = results

    print("\n" + "=" * 110)
    print(f"{disease_code} - {disease_name}")
    print("=" * 110)

    for rank, result in enumerate(results, start=1):
        print(
            f"{rank:>2}. "
            f"Score={result['similarity_score']:.4f} | "
            f"Section={result['section']} | "
            f"Chunk ID={result['chunk_id']}"
        )

        # Short preview to make comparison easier
        preview = " ".join(result["text"].split())
        print(f"    {preview[:250]}")

In [ ]:
# ============================================================
# Prepare Patient-RAG evidence for clinical concept extraction
# ============================================================

def format_patient_rag_evidence(
    disease_code,
    disease_name,
    retrieved_chunks
):
    """
    Format retrieved Patient-RAG chunks for the clinical
    evidence-extraction LLM.

    Important:
    Retrieval relevance does NOT imply that the disease is present.
    The chunks are preserved as raw patient evidence.
    """

    evidence_blocks = []

    for rank, chunk in enumerate(retrieved_chunks, start=1):

        evidence_blocks.append(
            f"[Chunk {rank}]\n"
            f"Section: {chunk['section']}\n"
            f"Text: {chunk['text']}"
        )

    evidence_text = "\n\n".join(evidence_blocks)

    return (
        f"Candidate condition: {disease_name} ({disease_code})\n\n"
        f"Retrieved patient evidence:\n\n"
        f"{evidence_text}"
    )

In [ ]:
hf_evidence_input = format_patient_rag_evidence(
    disease_code="CIR019",
    disease_name="Heart failure",
    retrieved_chunks=cardiology_results["CIR019"]
)

print(hf_evidence_input)

In [ ]:
# ============================================================
# MedGemma configuration
# ============================================================

from transformers import AutoTokenizer

MODEL_ID = "google/medgemma-4b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Tokenizer loaded successfully")
print("Tokenizer class:", type(tokenizer).__name__)
print("Model max length:", tokenizer.model_max_length)

In [ ]:
# ============================================================
# Load MedGemma for clinical evidence extraction
# ============================================================

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.eval()

print("MedGemma loaded successfully")
print("Model device:", model.device)

In [ ]:
# ============================================================
# Concept-focused summary prompt for KB retrieval
# ============================================================

def build_evidence_extraction_prompt(
    disease_name,
    disease_code,
    formatted_patient_evidence
):
    return f"""
You are creating a concise clinical semantic summary for
medical knowledge-base retrieval.

Candidate condition: {disease_name} ({disease_code})

IMPORTANT:
The candidate condition is a retrieval hypothesis only.
Do NOT assume that the patient has this condition.
Do NOT make a diagnosis.

Using ONLY the patient evidence provided below, summarize the
clinical concepts relevant to assessing or managing the candidate
condition.

Rules:
- Include only concepts explicitly supported by the patient evidence.
- Focus on clinically meaningful concepts useful for guideline retrieval.
- Include relevant symptoms and signs.
- Include relevant laboratory concepts without numerical values.
- Include relevant imaging findings.
- Include relevant comorbidities and organ dysfunction.
- Include relevant documented treatments or therapies.
- Include competing conditions when they may affect interpretation.
- Do NOT include patient-specific numerical values.
- Do NOT include age or other demographic details.
- Do NOT invent or infer findings.
- Do NOT generate new recommendations.
- Do NOT convert planned/recommended management into completed treatment.
- Do NOT include generic categories such as "labs", "symptoms",
  "medical history", or "imaging findings".
- Avoid repetition.
- Write one concise clinical paragraph.
- The summary will be used only as a semantic query for
  knowledge-base retrieval.
- Describe documented findings and conditions without deciding whether
  the candidate condition is currently present, absent, acute,
  decompensated, exacerbated, or otherwise diagnostically classified.

Return ONLY the clinical semantic summary.

PATIENT EVIDENCE:
{formatted_patient_evidence}
""".strip()

In [ ]:
# ============================================================
# Clinical evidence extraction with MedGemma
# ============================================================

def extract_clinical_evidence(
    extraction_prompt,
    model,
    processor,
    max_new_tokens=512
):
    """
    Use MedGemma to extract concise clinical evidence from
    disease-specific Patient-RAG results.

    This step does NOT make the final diagnosis.
    Its output will be used to construct the KB-RAG query.
    """

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": extraction_prompt
                }
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    input_length = inputs["input_ids"].shape[-1]

    print(
        f"Evidence extraction input: "
        f"{input_length:,} tokens"
    )

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=4
        )

    generated_tokens = outputs[0][input_length:]

    extracted_evidence = processor.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return {
        "extracted_evidence": extracted_evidence,
        "input_token_count": input_length,
        "generated_token_count": len(generated_tokens),
        "hit_generation_limit": (
            len(generated_tokens) >= max_new_tokens
        )
    }

In [ ]:
# Building the prompt according to the specialty and selected disease- part of the prompt is the raw patient data
hf_extraction_prompt = build_evidence_extraction_prompt(
    disease_name="Heart failure",
    disease_code="CIR019",
    formatted_patient_evidence=hf_evidence_input
)

# Actually extracting the summary from the patient raw data (included in the prompt) using the prompt 
hf_extraction_result = extract_clinical_evidence(
    hf_extraction_prompt,
    model,
    processor,
    max_new_tokens=512
)

print("\n===== GROUNDED EVIDENCE SUMMARY =====\n")
print(hf_extraction_result["extracted_evidence"])

print("\n===== GENERATION INFO =====")
print("Input tokens:", hf_extraction_result["input_token_count"])
print("Generated tokens:", hf_extraction_result["generated_token_count"])
print("Hit generation limit:", hf_extraction_result["hit_generation_limit"])

In [ ]:
# ============================================================
# Embed the MedGemma semantic summary for KB retrieval
# ============================================================

hf_semantic_summary = hf_extraction_result["extracted_evidence"]

hf_query_embedding = embedding_model.encode(
    hf_semantic_summary,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Semantic summary:")
print(hf_semantic_summary)

print("\nEmbedding shape:", hf_query_embedding.shape)
print("Embedding dtype:", hf_query_embedding.dtype)
print("Embedding norm:", np.linalg.norm(hf_query_embedding))

In [ ]:
# Loading the cardiology KB chunks:
import json
from pathlib import Path

KB_CHUNKS_FILE = Path(
    "Knowledge Base/Processed/cardiology_chunks.jsonl"
)

with KB_CHUNKS_FILE.open("r", encoding="utf-8") as f:
    cardiology_kb_chunks = [
        json.loads(line) for line in f
    ]

print(
    f"Loaded {len(cardiology_kb_chunks)} "
    f"Cardiology KB chunks."
)

In [ ]:
# Loading the embeding of the cardiology KB chunks
# ============================================================
# Load Cardiology KB BGE embeddings
# ============================================================

import numpy as np
from pathlib import Path

KB_EMBEDDINGS_FILE = Path(
    "Knowledge Base/Embeddings/cardiology_bge-base-en-v1.5.npz"
)

cardiology_embedding_data = np.load(
    KB_EMBEDDINGS_FILE,
    allow_pickle=True
)

print("Stored arrays:", cardiology_embedding_data.files)

cardiology_kb_embeddings = cardiology_embedding_data["embeddings"]
cardiology_embedding_chunk_ids = cardiology_embedding_data["chunk_ids"]

print("Embeddings shape:", cardiology_kb_embeddings.shape)
print("Embeddings dtype:", cardiology_kb_embeddings.dtype)
print("Number of chunk IDs:", len(cardiology_embedding_chunk_ids))

In [ ]:
# let's verify that the JSONL chunks and embedding rows are aligned exactly:
# ============================================================
# Verify Cardiology KB chunk ↔ embedding alignment
# ============================================================

jsonl_chunk_ids = [
    chunk["chunk_id"]
    for chunk in cardiology_kb_chunks
]

embedding_chunk_ids = [
    str(chunk_id)
    for chunk_id in cardiology_embedding_chunk_ids
]

print("JSONL chunks:", len(jsonl_chunk_ids))
print("Embedding IDs:", len(embedding_chunk_ids))

print(
    "Exact order match:",
    jsonl_chunk_ids == embedding_chunk_ids
)

print("\nFirst 3 JSONL IDs:")
print(jsonl_chunk_ids[:3])

print("\nFirst 3 embedding IDs:")
print(embedding_chunk_ids[:3])

In [ ]:
# ============================================================
# Verify Cardiology KB embeddings are normalized
# ============================================================

kb_embedding_norms = np.linalg.norm(
    cardiology_kb_embeddings,
    axis=1
)

print("Minimum norm:", kb_embedding_norms.min())
print("Maximum norm:", kb_embedding_norms.max())
print("Mean norm:", kb_embedding_norms.mean())

print(
    "All approximately unit-normalized:",
    np.allclose(kb_embedding_norms, 1.0, atol=1e-5)
)

In [ ]:
# ============================================================
# Retrieve top Cardiology KB chunks for HF semantic summary
# ============================================================

TOP_K_KB = 10

# Cosine similarity with all Cardiology KB chunks
kb_similarities = (
    cardiology_kb_embeddings @ hf_query_embedding
)

# Indices of highest-scoring chunks
top_indices = np.argsort(kb_similarities)[::-1][:TOP_K_KB]

hf_kb_results = []

for rank, idx in enumerate(top_indices, start=1):
    chunk = cardiology_kb_chunks[idx].copy()
    chunk["similarity_score"] = float(kb_similarities[idx])
    hf_kb_results.append(chunk)

    print("\n" + "=" * 100)
    print(
        f"Rank {rank} | "
        f"Score={kb_similarities[idx]:.4f} | "
        f"Chunk ID={chunk['chunk_id']}"
    )
    print("Source:", chunk.get("source_file"))
    print("Headings:", chunk.get("headings"))
    print("-" * 100)
    print(chunk["text"][:1000])

In [ ]:
# ============================================================
# Generate semantic summaries for all Cardiology candidates
# ============================================================

cardiology_semantic_summaries = {}

for disease_code, disease_name in cardiology_diseases.items():

    # Raw Patient-RAG chunks for this candidate disease
    retrieved_chunks = cardiology_results[disease_code]

    # Format raw patient evidence
    formatted_evidence = format_patient_rag_evidence(
        disease_code=disease_code,
        disease_name=disease_name,
        retrieved_chunks=retrieved_chunks
    )

    # Build retrieval-oriented MedGemma prompt
    extraction_prompt = build_evidence_extraction_prompt(
        disease_name=disease_name,
        disease_code=disease_code,
        formatted_patient_evidence=formatted_evidence
    )

    # Generate semantic summary
    extraction_result = extract_clinical_evidence(
        extraction_prompt,
        model,
        processor,
        max_new_tokens=512
    )

    cardiology_semantic_summaries[disease_code] = (
        extraction_result["extracted_evidence"]
    )

    print("\n" + "=" * 110)
    print(f"{disease_code} - {disease_name}")
    print("=" * 110)
    print(cardiology_semantic_summaries[disease_code])

    print(
        f"\nGenerated tokens: "
        f"{extraction_result['generated_token_count']}"
    )

In [ ]:
# ============================================================
# Embed all Cardiology semantic summaries for KB retrieval
# ============================================================

cardiology_query_embeddings = {}

for disease_code, summary in cardiology_semantic_summaries.items():

    query_embedding = embedding_model.encode(
        summary,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    cardiology_query_embeddings[disease_code] = query_embedding

    print(
        f"{disease_code}: "
        f"shape={query_embedding.shape}, "
        f"norm={np.linalg.norm(query_embedding):.6f}"
    )

In [ ]:
# ============================================================
# KB-RAG retrieval for all Cardiology candidates
# ============================================================

TOP_K_KB = 10

cardiology_kb_results = {}

for disease_code, disease_name in cardiology_diseases.items():

    query_embedding = cardiology_query_embeddings[disease_code]

    # Cosine similarity against all Cardiology KB chunks
    similarities = (
        cardiology_kb_embeddings @ query_embedding
    )

    # Retrieve top-k
    top_indices = np.argsort(similarities)[::-1][:TOP_K_KB]

    results = []

    for idx in top_indices:
        chunk = cardiology_kb_chunks[idx].copy()
        chunk["similarity_score"] = float(similarities[idx])
        results.append(chunk)

    cardiology_kb_results[disease_code] = results

    # Display retrieval results
    print("\n" + "=" * 110)
    print(f"{disease_code} - {disease_name}")
    print("=" * 110)

    for rank, result in enumerate(results, start=1):
        print(
            f"{rank:>2}. "
            f"Score={result['similarity_score']:.4f} | "
            f"Chunk ID={result['chunk_id']}"
        )
        print(f"    Source: {result.get('source_file')}")
        print(f"    Headings: {result.get('headings')}")

        preview = " ".join(result["text"].split())
        print(f"    {preview[:300]}")

In [ ]:
# ============================================================
# Prepare Cardiology evidence by candidate disease
# ============================================================

cardiology_evidence = {}

for disease_code, disease_name in cardiology_diseases.items():

    cardiology_evidence[disease_code] = {
        "disease_name": disease_name,

        # Original raw evidence retrieved from the patient case
        "patient_chunks": cardiology_results[disease_code],

        # Guideline evidence retrieved from the Cardiology KB
        "guideline_chunks": cardiology_kb_results[disease_code]
    }

# Quick verification
for disease_code, evidence in cardiology_evidence.items():

    print(
        f"{disease_code} - {evidence['disease_name']} | "
        f"Patient chunks: {len(evidence['patient_chunks'])} | "
        f"Guideline chunks: {len(evidence['guideline_chunks'])}"
    )

In [ ]:
# ============================================================
# Format Cardiology evidence for the specialist
# ============================================================

def format_cardiology_evidence(cardiology_evidence):

    disease_blocks = []

    for disease_code, evidence in cardiology_evidence.items():

        disease_name = evidence["disease_name"]

        # ----------------------------
        # Raw patient evidence
        # ----------------------------
        patient_blocks = []

        for rank, chunk in enumerate(
            evidence["patient_chunks"],
            start=1
        ):
            patient_blocks.append(
                f"[Patient Evidence {rank}]\n"
                f"Section: {chunk['section']}\n"
                f"Text: {chunk['text']}"
            )

        patient_text = "\n\n".join(patient_blocks)

        # ----------------------------
        # Retrieved guideline evidence
        # ----------------------------
        guideline_blocks = []

        for rank, chunk in enumerate(
            evidence["guideline_chunks"],
            start=1
        ):
            guideline_blocks.append(
                f"[Guideline Evidence {rank}]\n"
                f"Source: {chunk.get('source_file')}\n"
                f"Headings: {chunk.get('headings')}\n"
                f"Text: {chunk['text']}"
            )

        guideline_text = "\n\n".join(guideline_blocks)

        # ----------------------------
        # Disease-specific block
        # ----------------------------
        disease_blocks.append(
            f"{'=' * 80}\n"
            f"Candidate condition: {disease_name} "
            f"({disease_code})\n"
            f"{'=' * 80}\n\n"
            f"RAW PATIENT EVIDENCE\n\n"
            f"{patient_text}\n\n"
            f"RETRIEVED GUIDELINE EVIDENCE\n\n"
            f"{guideline_text}"
        )

    return "\n\n".join(disease_blocks)

In [ ]:
# ============================================================
# Format and measure uncompressed Cardiology evidence
# ============================================================

cardiology_formatted_evidence = format_cardiology_evidence(
    cardiology_evidence
)

# Token count of evidence itself
evidence_tokens = processor.tokenizer(
    cardiology_formatted_evidence,
    add_special_tokens=False,
    return_tensors="pt"
)["input_ids"].shape[-1]

print(
    f"Uncompressed Cardiology evidence: "
    f"{evidence_tokens:,} tokens"
)

# Also show token count by candidate disease
print("\nToken count by candidate disease:")

for disease_code, evidence in cardiology_evidence.items():

    single_disease_evidence = {
        disease_code: evidence
    }

    formatted = format_cardiology_evidence(
        single_disease_evidence
    )

    token_count = processor.tokenizer(
        formatted,
        add_special_tokens=False,
        return_tensors="pt"
    )["input_ids"].shape[-1]

    print(
        f"{disease_code} - "
        f"{evidence['disease_name']}: "
        f"{token_count:,} tokens"
    )

In [ ]:
# ============================================================
# Measure Patient-RAG vs KB-RAG evidence separately
# ============================================================

def count_text_tokens(text):
    return processor.tokenizer(
        text,
        add_special_tokens=False,
        return_tensors="pt"
    )["input_ids"].shape[-1]


for disease_code, evidence in cardiology_evidence.items():

    # Patient evidence
    patient_text = "\n\n".join(
        [
            f"[Patient Evidence {rank}]\n"
            f"Section: {chunk['section']}\n"
            f"Text: {chunk['text']}"
            for rank, chunk in enumerate(
                evidence["patient_chunks"],
                start=1
            )
        ]
    )

    # Guideline evidence
    guideline_text = "\n\n".join(
        [
            f"[Guideline Evidence {rank}]\n"
            f"Source: {chunk.get('source_file')}\n"
            f"Headings: {chunk.get('headings')}\n"
            f"Text: {chunk['text']}"
            for rank, chunk in enumerate(
                evidence["guideline_chunks"],
                start=1
            )
        ]
    )

    patient_tokens = count_text_tokens(patient_text)
    guideline_tokens = count_text_tokens(guideline_text)

    print(
        f"{disease_code} | "
        f"Patient: {patient_tokens:,} | "
        f"Guidelines: {guideline_tokens:,} | "
        f"Total: {patient_tokens + guideline_tokens:,}"
    )

In [ ]:
# ============================================================
# Model 3 adaptive-context compression prompts
# ============================================================

def build_patient_compression_prompt(
    disease_code,
    disease_name,
    patient_evidence
):
    return f"""
You are performing evidence-preserving condensation of retrieved
patient records for a Cardiology specialist.

Candidate condition: {disease_name} ({disease_code})

The candidate condition is provided ONLY to identify which evidence
is relevant. Do NOT assume, diagnose, confirm, or reject the condition.

Your task is NOT to perform clinical reasoning or write a diagnostic
summary.

Your task is to reduce the retrieved patient evidence by:
1. retaining clinically relevant documented facts,
2. removing duplicated facts,
3. removing clearly irrelevant material,
while preserving the meaning of the original evidence.

RETAIN, when relevant:
- documented symptoms
- documented physical examination findings
- vital signs
- laboratory results
- imaging findings
- ECG and other diagnostic findings
- relevant medical history and comorbidities
- medications
- treatments
- procedures
- documented clinical course
- documented response to treatment
- relevant renal, respiratory, infectious, or other findings
- explicit negations
- uncertainty stated in the original evidence
- relevant temporal information

STRICT EVIDENCE-PRESERVATION RULES:
- Use ONLY facts explicitly present in the provided patient evidence.
- Do NOT generate a diagnosis or diagnostic conclusion.
- Do NOT state whether the candidate condition is present or absent.
- Do NOT synthesize several findings into a new clinical conclusion.
- Do NOT add causal relationships that are not explicitly documented.
- Do NOT add clinical interpretations to measurements.
- Do NOT introduce descriptive labels such as "elevated", "low",
  "abnormal", "impaired", "compromised", "concerning for",
  "consistent with", or "suggestive of" unless that interpretation
  is explicitly stated in the source evidence.
- Do NOT add recommendations.
- Do NOT convert planned or recommended treatment into completed treatment.
- Preserve source uncertainty and negation.

NUMERICAL FIDELITY:
- Retain a numerical value only when it is clearly represented in the
  source evidence and clinically relevant.
- When retaining a value, preserve the value and unit exactly as supported
  by the source evidence.
- Do NOT reconstruct, combine, reorder, normalize, round, or guess values.
- Do NOT create ranges or trends unless they are explicitly supported.
- If a numerical sequence is unclear or malformed, do not attempt to
  reconstruct it.
- In that situation, retain the associated clinical fact without inventing
  a numerical value, if that fact itself is explicitly supported.

COMPRESSION:
- Remove repeated occurrences of the same fact.
- Remove clearly irrelevant information.
- Do NOT remove a distinct clinically relevant fact merely to shorten
  the output.
- Prefer evidence fidelity over brevity.
- Keep different findings separate rather than combining them into a
  diagnostic interpretation.

Return ONLY the condensed patient evidence.
Do not include the candidate diagnosis name as a conclusion or heading.

PATIENT EVIDENCE:
{patient_evidence}
""".strip()


def build_guideline_compression_prompt(
    disease_code,
    disease_name,
    guideline_evidence
):
    return f"""
You are compressing retrieved clinical guideline evidence for a
Cardiology specialist.

Candidate condition: {disease_name} ({disease_code})

Compress ONLY the guideline evidence provided below.

Preserve clinically useful guideline information relevant to:
- diagnostic assessment
- differential diagnosis
- investigations
- treatment and management
- contraindications and precautions
- relevant comorbidities
- medication safety
- monitoring and follow-up
- clinically important thresholds or criteria

Rules:
- Use ONLY information explicitly present in the guideline evidence.
- Preserve important thresholds, criteria, doses, and conditions when relevant.
- Preserve uncertainty, qualifications, contraindications, and exceptions.
- Do NOT introduce recommendations that are absent from the retrieved evidence.
- Do NOT treat guideline statements as facts about the patient.
- Remove bibliographic/reference-list material when it does not itself
  contain clinically useful guidance.
- Remove repetition and material not useful for assessing or managing
  the candidate condition.
- Produce a concise guideline evidence summary.

Return ONLY the compressed guideline evidence.

GUIDELINE EVIDENCE:
{guideline_evidence}
""".strip()

In [ ]:
# ============================================================
# Compress one isolated Model 3 evidence unit
# ============================================================

def compress_model3_evidence_unit(
    evidence_text,
    disease_code,
    disease_name,
    evidence_type,
    model,
    processor,
    max_new_tokens=1024
):
    """
    Compress ONE evidence unit only.

    evidence_type must be:
        "patient"   -> Patient-RAG evidence for one disease
        "guideline" -> KB-RAG evidence for one disease

    Patient and guideline evidence are never mixed during compression.
    """

    if evidence_type == "patient":

        prompt = build_patient_compression_prompt(
            disease_code=disease_code,
            disease_name=disease_name,
            patient_evidence=evidence_text
        )

    elif evidence_type == "guideline":

        prompt = build_guideline_compression_prompt(
            disease_code=disease_code,
            disease_name=disease_name,
            guideline_evidence=evidence_text
        )

    else:
        raise ValueError(
            "evidence_type must be 'patient' or 'guideline'."
        )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    input_length = inputs["input_ids"].shape[-1]

    print(
        f"Compressing {disease_code} "
        f"({evidence_type}) | "
        f"formatted input: {input_length:,} tokens"
    )

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=4
        )

    generated_tokens = outputs[0][input_length:]

    compressed_text = processor.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return {
        "compressed_text": compressed_text,
        "input_token_count": input_length,
        "generated_token_count": len(generated_tokens),
        "hit_generation_limit": (
            len(generated_tokens) >= max_new_tokens
        )
    }

In [ ]:
#Originally, for each of the five Cardiology candidate diseases, we already had: 
#CIR007 – Essential hypertension
#├── Patient evidence
#└── Guideline evidence
# This function converted these into two explicitly independent units.
# So across five diseases, we will have:
# CIR017 patient
# CIR017 guideline

# CIR019 patient
# CIR019 guideline

# CIR008 patient
# CIR008 guideline

# CIR011 patient
# CIR011 guideline

# CIR007 patient
# CIR007 guideline

# ============================================================
# Initialise Model 3 adaptive-context state
# ============================================================

def initialise_cardiology_context_state(cardiology_evidence):

    context_state = {}

    for disease_code, evidence in cardiology_evidence.items():

        disease_name = evidence["disease_name"]

        # -------------------------
        # Patient evidence
        # -------------------------

        patient_blocks = []

        for rank, chunk in enumerate(
            evidence["patient_chunks"],
            start=1
        ):
            patient_blocks.append(
                f"[Patient Evidence {rank}]\n"
                f"Section: {chunk['section']}\n"
                f"Text: {chunk['text']}"
            )

        patient_text = "\n\n".join(patient_blocks)

        # -------------------------
        # Guideline evidence
        # -------------------------

        guideline_blocks = []

        for rank, chunk in enumerate(
            evidence["guideline_chunks"],
            start=1
        ):
            guideline_blocks.append(
                f"[Guideline Evidence {rank}]\n"
                f"Source: {chunk.get('source_file')}\n"
                f"Headings: {chunk.get('headings')}\n"
                f"Text: {chunk['text']}"
            )

        guideline_text = "\n\n".join(guideline_blocks)

        # -------------------------
        # Store two independent units
        # -------------------------

        context_state[(disease_code, "patient")] = {
            "disease_code": disease_code,
            "disease_name": disease_name,
            "evidence_type": "patient",
            "chunks": evidence["patient_chunks"],
            "text": patient_text,
            "token_count": count_text_tokens(patient_text),
            "compressed": False
        }

        context_state[(disease_code, "guideline")] = {
            "disease_code": disease_code,
            "disease_name": disease_name,
            "evidence_type": "guideline",
            "chunks": evidence["guideline_chunks"],
            "text": guideline_text,
            "token_count": count_text_tokens(guideline_text),
            "compressed": False
        }

    return context_state


cardiology_context_state = initialise_cardiology_context_state(
    cardiology_evidence
)


# Verify the 10 independent units
for key, unit in cardiology_context_state.items():
    print(
        f"{unit['disease_code']:6} | "
        f"{unit['evidence_type']:9} | "
        f"{unit['token_count']:6,} tokens | "
        f"compressed={unit['compressed']}"
    )

In [ ]:
# ============================================================
# Cardiology specialist prompt
# ============================================================

CARDIOLOGY_SPECIALIST_PROMPT = """
You are the Cardiology specialist in a multi-specialist clinical
reasoning system.

Your task is to assess the provided patient evidence together with
the retrieved cardiology guideline evidence and produce a
specialty-specific prediction at the end of the admission.

You may consider ONLY the following candidate diagnosis categories:

- CIR017: Cardiac dysrhythmias
- CIR019: Heart failure
- CIR008: Hypertension with complications and secondary hypertension
- CIR011: Coronary atherosclerosis and other heart disease
- CIR007: Essential hypertension

DIAGNOSIS RULES:
- Predict only diagnoses that are supported by the patient evidence.
- The presence of retrieved evidence for a candidate condition does
  NOT mean that the patient has that condition.
- Do not force a diagnosis from the candidate list.
- It is acceptable for none of the candidate diagnoses to be supported.
- Do not invent patient findings.
- Preserve uncertainty when the evidence is uncertain.
- Distinguish active/relevant conditions from historical conditions
  when the evidence permits.
- Use guideline evidence to support clinical reasoning, but never treat
  a guideline statement as a fact about the patient.

MANAGEMENT RULES:
- Predict discharge/end-of-admission management relevant to the
  supported Cardiology diagnoses and patient context.
- Management must be grounded in the provided patient evidence and
  retrieved guideline evidence.
- Consider relevant medications, treatment changes, monitoring,
  follow-up, procedures, precautions, and contraindications when
  supported.
- Do not assume that a guideline recommendation was actually performed
  unless the patient evidence states that it was.
- Do not invent treatments, investigations, or follow-up.

Return the result using exactly this structure:

SUPPORTED CARDIOLOGY DIAGNOSES:
- <candidate code>: <candidate diagnosis>
or
- None supported

CARDIOLOGY DISCHARGE MANAGEMENT:
- <management item>
or
- No specific Cardiology discharge management supported

Briefly include the evidence basis within each diagnosis or management
item where useful.
""".strip()

In [ ]:
# ============================================================
# Build and measure complete Cardiology specialist input
# ============================================================

OPERATIONAL_INPUT_LIMIT = 20000


def build_cardiology_context_text(context_state):

    disease_blocks = []

    for disease_code, disease_name in cardiology_diseases.items():

        patient_unit = context_state[
            (disease_code, "patient")
        ]

        guideline_unit = context_state[
            (disease_code, "guideline")
        ]

        disease_blocks.append(
            f"{'=' * 80}\n"
            f"Candidate condition: {disease_name} "
            f"({disease_code})\n"
            f"{'=' * 80}\n\n"

            f"PATIENT EVIDENCE\n"
            f"{patient_unit['text']}\n\n"

            f"GUIDELINE EVIDENCE\n"
            f"{guideline_unit['text']}"
        )

    return "\n\n".join(disease_blocks)


def measure_cardiology_specialist_input(
    context_state,
    processor
):

    evidence_text = build_cardiology_context_text(
        context_state
    )

    full_prompt = (
        CARDIOLOGY_SPECIALIST_PROMPT
        + "\n\n"
        + "CARDIOLOGY EVIDENCE:\n\n"
        + evidence_text
    )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": full_prompt
                }
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    input_length = inputs["input_ids"].shape[-1]

    return {
        "evidence_text": evidence_text,
        "full_prompt": full_prompt,
        "input_token_count": input_length,
        "within_limit": (
            input_length <= OPERATIONAL_INPUT_LIMIT
        )
    }


cardiology_input_check = measure_cardiology_specialist_input(
    cardiology_context_state,
    processor
)

print(
    f"Complete formatted Cardiology input: "
    f"{cardiology_input_check['input_token_count']:,} tokens"
)

print(
    f"Operational limit: "
    f"{OPERATIONAL_INPUT_LIMIT:,} tokens"
)

print(
    f"Within limit: "
    f"{cardiology_input_check['within_limit']}"
)

In [ ]:
# ============================================================
# Select largest uncompressed evidence unit
# ============================================================

def select_largest_uncompressed_unit(context_state):

    eligible_units = [
        unit
        for unit in context_state.values()
        if not unit["compressed"]
    ]

    if not eligible_units:
        return None

    return max(
        eligible_units,
        key=lambda unit: unit["token_count"]
    )


largest_unit = select_largest_uncompressed_unit(
    cardiology_context_state
)

print(
    f"Selected unit: "
    f"{largest_unit['disease_code']} | "
    f"{largest_unit['evidence_type']}"
)

print(
    f"Current size: "
    f"{largest_unit['token_count']:,} tokens"
)

In [ ]:
# ============================================================
# Check whether one evidence unit fits one compression call
# ============================================================

# Why 12,000 rather than 20,000?
# The 20,000 limit is our operational limit for the final specialist input.
# The compression operation is different: it needs room for the compression instructions and for generated output, 
# so I'm deliberately giving it a more conservative input ceiling. We can keep this as a fixed methodological parameter across all cases.
COMPRESSION_INPUT_LIMIT = 12000


def measure_model3_compression_input(
    unit,
    processor
):
    """
    Measure the complete formatted MedGemma input required to
    compress ONE disease/source evidence unit.

    No compression is performed here.
    """

    if unit["evidence_type"] == "patient":

        prompt = build_patient_compression_prompt(
            disease_code=unit["disease_code"],
            disease_name=unit["disease_name"],
            patient_evidence=unit["text"]
        )

    elif unit["evidence_type"] == "guideline":

        prompt = build_guideline_compression_prompt(
            disease_code=unit["disease_code"],
            disease_name=unit["disease_name"],
            guideline_evidence=unit["text"]
        )

    else:
        raise ValueError(
            "evidence_type must be 'patient' or 'guideline'."
        )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    input_length = inputs["input_ids"].shape[-1]

    return {
        "input_token_count": input_length,
        "fits_single_call": (
            input_length <= COMPRESSION_INPUT_LIMIT
        )
    }

###############################################################
compression_check = measure_model3_compression_input(
    largest_unit,
    processor
)

print(
    f"Selected unit: "
    f"{largest_unit['disease_code']} | "
    f"{largest_unit['evidence_type']}"
)

print(
    f"Raw unit size: "
    f"{largest_unit['token_count']:,} tokens"
)

print(
    f"Formatted compression input: "
    f"{compression_check['input_token_count']:,} tokens"
)

print(
    f"Compression input limit: "
    f"{COMPRESSION_INPUT_LIMIT:,} tokens"
)

print(
    f"Fits single compression call: "
    f"{compression_check['fits_single_call']}"
)

In [ ]:
# ============================================================
# Measure exact Patient-RAG chunk duplication across
# the five Cardiology candidate diseases
# ============================================================

from collections import Counter, defaultdict

chunk_usage = Counter()
chunk_diseases = defaultdict(list)

for disease_code, evidence in cardiology_evidence.items():
    for chunk in evidence["patient_chunks"]:
        chunk_id = chunk["chunk_id"]

        chunk_usage[chunk_id] += 1
        chunk_diseases[chunk_id].append(disease_code)


total_retrieved = sum(chunk_usage.values())
unique_chunks = len(chunk_usage)

duplicate_occurrences = total_retrieved - unique_chunks

chunks_shared_across_diseases = {
    chunk_id: diseases
    for chunk_id, diseases in chunk_diseases.items()
    if len(diseases) > 1
}


print(f"Total retrieved patient chunks: {total_retrieved}")
print(f"Unique patient chunks: {unique_chunks}")
print(f"Duplicate occurrences: {duplicate_occurrences}")

print(
    f"Chunks shared across candidate diseases: "
    f"{len(chunks_shared_across_diseases)}"
)

print(
    f"Exact duplication reduction: "
    f"{duplicate_occurrences / total_retrieved:.1%}"
)

print("\nShared chunks:")
for chunk_id, diseases in chunks_shared_across_diseases.items():
    print(
        f"{chunk_id} | "
        f"retrieved for {len(diseases)} diseases: "
        f"{', '.join(diseases)}"
    )

In [ ]:
# ============================================================
# Measure token saving from exact Patient-RAG deduplication
# ============================================================

unique_patient_chunks = []
seen_chunk_ids = set()

for disease_code, evidence in cardiology_evidence.items():
    for chunk in evidence["patient_chunks"]:

        chunk_id = chunk["chunk_id"]

        if chunk_id not in seen_chunk_ids:
            seen_chunk_ids.add(chunk_id)
            unique_patient_chunks.append(chunk)


unique_patient_blocks = []

for rank, chunk in enumerate(unique_patient_chunks, start=1):
    unique_patient_blocks.append(
        f"[Patient Evidence {rank}]\n"
        f"Section: {chunk['section']}\n"
        f"Text: {chunk['text']}"
    )

unique_patient_text = "\n\n".join(unique_patient_blocks)

unique_patient_tokens = count_text_tokens(
    unique_patient_text
)

original_patient_tokens = sum(
    cardiology_context_state[
        (disease_code, "patient")
    ]["token_count"]
    for disease_code in cardiology_diseases
)

print(
    f"Original disease-separated patient evidence: "
    f"{original_patient_tokens:,} tokens"
)

print(
    f"Deduplicated patient evidence: "
    f"{unique_patient_tokens:,} tokens"
)

print(
    f"Tokens removed: "
    f"{original_patient_tokens - unique_patient_tokens:,}"
)

print(
    f"Token reduction: "
    f"{1 - unique_patient_tokens / original_patient_tokens:.1%}"
)

In [ ]:
# ============================================================
# Measure complete Cardiology input after lossless
# Patient-RAG deduplication
# KB evidence remains unchanged
# ============================================================

def build_deduplicated_cardiology_context(
    unique_patient_text,
    cardiology_context_state
):
    guideline_blocks = []

    for disease_code, disease_name in cardiology_diseases.items():

        guideline_unit = cardiology_context_state[
            (disease_code, "guideline")
        ]

        guideline_blocks.append(
            f"{'=' * 80}\n"
            f"Candidate condition: {disease_name} "
            f"({disease_code})\n"
            f"{'=' * 80}\n\n"
            f"GUIDELINE EVIDENCE\n"
            f"{guideline_unit['text']}"
        )

    guideline_text = "\n\n".join(guideline_blocks)

    return (
        "DEDUPLICATED RAW PATIENT EVIDENCE\n\n"
        + unique_patient_text
        + "\n\n"
        + "DISEASE-SPECIFIC GUIDELINE EVIDENCE\n\n"
        + guideline_text
    )


deduplicated_context_text = build_deduplicated_cardiology_context(
    unique_patient_text,
    cardiology_context_state
)

deduplicated_full_prompt = (
    CARDIOLOGY_SPECIALIST_PROMPT
    + "\n\n"
    + "CARDIOLOGY EVIDENCE:\n\n"
    + deduplicated_context_text
)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": deduplicated_full_prompt
            }
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
)

deduplicated_complete_tokens = inputs["input_ids"].shape[-1]

print(
    f"Original complete Cardiology input: "
    f"43,750 tokens"
)

print(
    f"After patient deduplication: "
    f"{deduplicated_complete_tokens:,} tokens"
)

print(
    f"Operational limit: "
    f"{OPERATIONAL_INPUT_LIMIT:,} tokens"
)

print(
    f"Still above limit by: "
    f"{max(0, deduplicated_complete_tokens - OPERATIONAL_INPUT_LIMIT):,} tokens"
)

print(
    f"Within limit: "
    f"{deduplicated_complete_tokens <= OPERATIONAL_INPUT_LIMIT}"
)

In [ ]:
# ============================================================
# Select largest uncompressed GUIDELINE unit only
# Patient evidence is never selected for compression
# ============================================================

def select_largest_uncompressed_guideline_unit(context_state):

    eligible_units = [
        unit
        for unit in context_state.values()
        if (
            unit["evidence_type"] == "guideline"
            and not unit["compressed"]
        )
    ]

    if not eligible_units:
        return None

    return max(
        eligible_units,
        key=lambda unit: unit["token_count"]
    )


largest_guideline_unit = (
    select_largest_uncompressed_guideline_unit(
        cardiology_context_state
    )
)

if largest_guideline_unit is None:
    print("No uncompressed guideline units remain.")

else:
    print(
        f"Selected guideline unit: "
        f"{largest_guideline_unit['disease_code']} | "
        f"{largest_guideline_unit['disease_name']}"
    )

    print(
        f"Current size: "
        f"{largest_guideline_unit['token_count']:,} tokens"
    )

In [ ]:
# ============================================================
# Test one-time compression of selected guideline unit
# DO NOT update context state yet
# ============================================================

guideline_compression_test = compress_model3_evidence_unit(
    evidence_text=largest_guideline_unit["text"],
    disease_code=largest_guideline_unit["disease_code"],
    disease_name=largest_guideline_unit["disease_name"],
    evidence_type="guideline",
    model=model,
    processor=processor,
    max_new_tokens=1024
)

print(
    f"Original guideline evidence: "
    f"{largest_guideline_unit['token_count']:,} tokens"
)

print(
    f"Formatted compression input: "
    f"{guideline_compression_test['input_token_count']:,} tokens"
)

print(
    f"Compressed guideline evidence: "
    f"{guideline_compression_test['generated_token_count']:,} tokens"
)

print(
    f"Hit generation limit: "
    f"{guideline_compression_test['hit_generation_limit']}"
)

print("\n" + "=" * 80)
print("COMPRESSED GUIDELINE EVIDENCE")
print("=" * 80)

print(
    guideline_compression_test["compressed_text"]
)

In [ ]:
# ============================================================
# Inspect original CIR011 KB-RAG retrieval
# ============================================================

cir011_kb_results = cardiology_kb_results["CIR011"]

print(
    f"CIR011 retrieved guideline chunks: "
    f"{len(cir011_kb_results)}"
)

print("=" * 100)

for rank, chunk in enumerate(cir011_kb_results, start=1):

    print(f"\nRANK {rank}")
    print(
        f"Similarity score: "
        f"{chunk.get('similarity_score', 'N/A')}"
    )
    print(
        f"Source: "
        f"{chunk.get('source_file', 'N/A')}"
    )
    print(
        f"Headings: "
        f"{chunk.get('headings', 'N/A')}"
    )
    print(
        f"Chunk ID: "
        f"{chunk.get('chunk_id', 'N/A')}"
    )

    print("\nTEXT:")
    print(chunk["text"])

    print("\n" + "-" * 100)

In [ ]:
# ============================================================
# Check disease-specific guideline coverage in Cardiology KB
# ============================================================

for disease_code, disease_name in cardiology_diseases.items():

    matching_chunks = [
        chunk
        for chunk in cardiology_kb_chunks
        if disease_code in str(chunk.get("source_file", ""))
    ]

    matching_sources = sorted({
        chunk.get("source_file")
        for chunk in matching_chunks
    })

    print("=" * 80)
    print(f"{disease_code} | {disease_name}")
    print(f"Matching KB chunks: {len(matching_chunks)}")
    print(f"Matching source files: {len(matching_sources)}")

    for source in matching_sources:
        print(f"  - {source}")

In [ ]:
# ============================================================
# Build disease-specific Cardiology KB indices
# ============================================================

cardiology_disease_kb_indices = {}

for disease_code in cardiology_diseases:

    matching_indices = [
        i
        for i, chunk in enumerate(cardiology_kb_chunks)
        if disease_code in str(chunk.get("source_file", ""))
    ]

    cardiology_disease_kb_indices[disease_code] = matching_indices

    print(
        f"{disease_code}: "
        f"{len(matching_indices):,} eligible KB chunks"
    )
    

In [ ]:
# ============================================================
# Disease-specific Cardiology KB retrieval
# ============================================================

def retrieve_disease_specific_kb_chunks(
    disease_code,
    query_embedding,
    top_k=10
):
    eligible_indices = cardiology_disease_kb_indices[disease_code]

    if not eligible_indices:
        raise ValueError(
            f"No KB chunks available for {disease_code}"
        )

    eligible_embeddings = cardiology_kb_embeddings[
        eligible_indices
    ]

    similarities = eligible_embeddings @ query_embedding

    k = min(top_k, len(eligible_indices))
    top_local_indices = np.argsort(similarities)[::-1][:k]

    results = []

    for local_idx in top_local_indices:

        global_idx = eligible_indices[local_idx]

        result = cardiology_kb_chunks[global_idx].copy()
        result["similarity_score"] = float(
            similarities[local_idx]
        )

        results.append(result)

    return results


cardiology_kb_results_disease_specific = {}

for disease_code in cardiology_diseases:

    cardiology_kb_results_disease_specific[disease_code] = (
        retrieve_disease_specific_kb_chunks(
            disease_code=disease_code,
            query_embedding=cardiology_query_embeddings[
                disease_code
            ],
            top_k=10
        )
    )

    print(
        f"{disease_code}: "
        f"{len(cardiology_kb_results_disease_specific[disease_code])} "
        f"chunks retrieved"
    )

In [ ]:
# ============================================================
# Inspect disease-specific CIR011 KB retrieval
# ============================================================

cir011_new_results = (
    cardiology_kb_results_disease_specific["CIR011"]
)

print(
    f"CIR011 disease-specific retrieved chunks: "
    f"{len(cir011_new_results)}"
)

print("=" * 100)

for rank, chunk in enumerate(cir011_new_results, start=1):

    print(f"\nRANK {rank}")
    print(
        f"Similarity score: "
        f"{chunk.get('similarity_score', 'N/A')}"
    )
    print(
        f"Source: "
        f"{chunk.get('source_file', 'N/A')}"
    )
    print(
        f"Headings: "
        f"{chunk.get('headings', 'N/A')}"
    )
    print(
        f"Chunk ID: "
        f"{chunk.get('chunk_id', 'N/A')}"
    )

    print("\nTEXT:")
    print(chunk["text"])

    print("\n" + "-" * 100)

In [ ]:
# ============================================================
# Compare old vs disease-specific KB retrieval token sizes
# ============================================================

for disease_code, disease_name in cardiology_diseases.items():

    old_text = "\n\n".join(
        chunk["text"]
        for chunk in cardiology_kb_results[disease_code]
    )

    new_text = "\n\n".join(
        chunk["text"]
        for chunk in cardiology_kb_results_disease_specific[
            disease_code
        ]
    )

    old_tokens = len(
        tokenizer.encode(
            old_text,
            add_special_tokens=False
        )
    )

    new_tokens = len(
        tokenizer.encode(
            new_text,
            add_special_tokens=False
        )
    )

    print(
        f"{disease_code} | {disease_name}\n"
        f"  Old KB retrieval: {old_tokens:,} tokens\n"
        f"  Disease-specific: {new_tokens:,} tokens\n"
    )

In [ ]:
# ============================================================
# Rebuild deduplicated raw patient evidence
# + measure complete input with disease-specific KB retrieval
# ============================================================

# 1. Deduplicate patient chunks by chunk_id
unique_patient_chunks = {}

for disease_code in cardiology_diseases:

    for chunk in cardiology_results[disease_code]:

        chunk_id = chunk["chunk_id"]

        if chunk_id not in unique_patient_chunks:
            unique_patient_chunks[chunk_id] = {
                "chunk": chunk,
                "retrieved_for": []
            }

        unique_patient_chunks[chunk_id][
            "retrieved_for"
        ].append(disease_code)


# 2. Format each unique patient chunk once
patient_blocks = []

for item in unique_patient_chunks.values():

    chunk = item["chunk"]
    retrieved_for = ", ".join(item["retrieved_for"])

    patient_blocks.append(
        f"Relevant to retrieval queries: {retrieved_for}\n"
        f"Section: {chunk['section']}\n"
        f"Text: {chunk['text']}"
    )

deduplicated_patient_evidence_text = "\n\n".join(
    patient_blocks
)


# 3. Format disease-specific guideline evidence
guideline_blocks = []

for disease_code, disease_name in cardiology_diseases.items():

    chunks = (
        cardiology_kb_results_disease_specific[
            disease_code
        ]
    )

    guideline_text = "\n\n".join(
        chunk["text"]
        for chunk in chunks
    )

    guideline_blocks.append(
        f"Candidate: {disease_code} | {disease_name}\n\n"
        f"{guideline_text}"
    )

guideline_evidence_disease_specific = "\n\n".join(
    guideline_blocks
)


# 4. Build complete specialist input
complete_input_disease_specific = (
    CARDIOLOGY_SPECIALIST_PROMPT
    + "\n\nPATIENT EVIDENCE:\n"
    + deduplicated_patient_evidence_text
    + "\n\nGUIDELINE EVIDENCE:\n"
    + guideline_evidence_disease_specific
)


# 5. Measure
complete_tokens_disease_specific = len(
    tokenizer.encode(
        complete_input_disease_specific,
        add_special_tokens=False
    )
)

patient_tokens = len(
    tokenizer.encode(
        deduplicated_patient_evidence_text,
        add_special_tokens=False
    )
)

guideline_tokens = len(
    tokenizer.encode(
        guideline_evidence_disease_specific,
        add_special_tokens=False
    )
)


print(
    f"Unique patient chunks: "
    f"{len(unique_patient_chunks)}"
)

print(
    f"Deduplicated patient evidence: "
    f"{patient_tokens:,} tokens"
)

print(
    f"Disease-specific guideline evidence: "
    f"{guideline_tokens:,} tokens"
)

print(
    f"Complete Cardiology input: "
    f"{complete_tokens_disease_specific:,} tokens"
)

print("Operational limit: 20,000 tokens")

print(
    f"Tokens above limit: "
    f"{max(0, complete_tokens_disease_specific - 20000):,}"
)

print(
    f"Within limit: "
    f"{complete_tokens_disease_specific <= 20000}"
)

In [ ]:
# ============================================================
# Measure disease-specific guideline units
# ============================================================

cardiology_guideline_units = {}

for disease_code, disease_name in cardiology_diseases.items():

    guideline_text = "\n\n".join(
        chunk["text"]
        for chunk in cardiology_kb_results_disease_specific[
            disease_code
        ]
    )

    token_count = len(
        tokenizer.encode(
            guideline_text,
            add_special_tokens=False
        )
    )

    cardiology_guideline_units[disease_code] = {
        "disease_code": disease_code,
        "disease_name": disease_name,
        "text": guideline_text,
        "token_count": token_count,
        "compressed": False
    }


for unit in sorted(
    cardiology_guideline_units.values(),
    key=lambda x: x["token_count"],
    reverse=True
):
    print(
        f"{unit['disease_code']} | "
        f"{unit['disease_name']}: "
        f"{unit['token_count']:,} tokens"
    )

In [ ]:
# ============================================================
# Test compression of NEW disease-specific CIR011 guideline
# Do NOT update the guideline unit yet
# ============================================================

largest_guideline_unit = cardiology_guideline_units["CIR011"]

cir011_guideline_compression_test = compress_model3_evidence_unit(
    evidence_text=largest_guideline_unit["text"],
    disease_code=largest_guideline_unit["disease_code"],
    disease_name=largest_guideline_unit["disease_name"],
    evidence_type="guideline",
    model=model,
    processor=processor,
    max_new_tokens=1024
)

print(
    f"Original guideline evidence: "
    f"{largest_guideline_unit['token_count']:,} tokens"
)

print(
    f"Compressed guideline evidence: "
    f"{cir011_guideline_compression_test['generated_token_count']:,} tokens"
)

print(
    f"Hit generation limit: "
    f"{cir011_guideline_compression_test['hit_generation_limit']}"
)

print("\n" + "=" * 80)
print("COMPRESSED CIR011 GUIDELINE EVIDENCE")
print("=" * 80)

print(
    cir011_guideline_compression_test["compressed_text"]
)

In [ ]:
# ============================================================
# Disease-level assessment prompt aligned with discharge target
# ============================================================

CARDIOLOGY_DISEASE_ASSESSMENT_PROMPT = """
You are assessing ONE candidate Cardiology diagnosis at the
end of the patient's admission.

The candidate diagnosis is explicitly provided below.

Your task is to determine whether the supplied patient evidence
supports this candidate as an active or clinically relevant diagnosis
at the end of admission.

The candidate does NOT need to be the primary reason for admission
and does NOT need to explain the patient's acute presentation.

Use the retrieved guideline evidence only as medical knowledge to
help interpret the patient evidence and identify management
considerations.

Rules:
- Assess ONLY the specified candidate diagnosis.
- Do not assess or generate other diagnoses.
- The candidate is a hypothesis, not an assumed diagnosis.
- Retrieved evidence may contain irrelevant information.
- Use only patient evidence that is relevant to this candidate.
- Historical evidence may support a known chronic diagnosis, but
  clearly distinguish historical-only evidence from evidence showing
  current clinical relevance.
- Do not invent or alter patient findings.
- Do not assume that a guideline recommendation was performed.
- Do not present guideline recommendations as documented patient
  management.

Return ONLY:

ASSESSMENT: <Supported / Not supported / Uncertain>

RATIONALE:
<Briefly explain why the candidate is or is not supported as an
active or clinically relevant diagnosis at the end of admission.
Distinguish historical-only evidence when relevant.>

DOCUMENTED MANAGEMENT:
<Management for this candidate explicitly documented in the patient
evidence, or "None identified.">

GUIDELINE-SUPPORTED CONSIDERATIONS:
<Management considerations from the retrieved guideline evidence
that are applicable to the patient's documented context, or
"None identified.">

Stop after GUIDELINE-SUPPORTED CONSIDERATIONS.
Do not discuss any other diagnosis.
Do not generate or repeat a diagnosis code.
""".strip()

In [ ]:
# ============================================================
# Build complete disease-level Cardiology input
# ============================================================

def build_cardiology_disease_input(
    disease_code,
    disease_name,
    patient_chunks,
    guideline_chunks
):

    patient_evidence = "\n\n".join(
        f"[Patient Chunk {rank}]\n"
        f"Section: {chunk['section']}\n"
        f"Text: {chunk['text']}"
        for rank, chunk in enumerate(
            patient_chunks,
            start=1
        )
    )

    guideline_evidence = "\n\n".join(
        f"[Guideline Chunk {rank}]\n"
        f"Source: {chunk.get('source_file', 'N/A')}\n"
        f"Headings: {chunk.get('headings', 'N/A')}\n"
        f"Text: {chunk['text']}"
        for rank, chunk in enumerate(
            guideline_chunks,
            start=1
        )
    )

    return (
        CARDIOLOGY_DISEASE_ASSESSMENT_PROMPT
        + "\n\n"
        + "=" * 80
        + f"\n\nCANDIDATE TO ASSESS:\n"
        + f"{disease_code}: {disease_name}\n\n"
        + "=" * 80
        + "\n\nRAW PATIENT EVIDENCE:\n\n"
        + patient_evidence
        + "\n\n"
        + "=" * 80
        + "\n\nRAW GUIDELINE EVIDENCE:\n\n"
        + guideline_evidence
    )

In [ ]:
# ============================================================
# Build and measure one disease-level input: CIR011
# ============================================================

test_disease_code = "CIR011"
test_disease_name = cardiology_diseases[test_disease_code]

cir011_disease_input = build_cardiology_disease_input(
    disease_code=test_disease_code,
    disease_name=test_disease_name,
    patient_chunks=cardiology_results[test_disease_code],
    guideline_chunks=cardiology_kb_results_disease_specific[
        test_disease_code
    ]
)

cir011_input_tokens = len(
    tokenizer.encode(
        cir011_disease_input,
        add_special_tokens=False
    )
)

print(f"Disease: {test_disease_code} | {test_disease_name}")
print(
    f"Patient chunks: "
    f"{len(cardiology_results[test_disease_code])}"
)
print(
    f"Guideline chunks: "
    f"{len(cardiology_kb_results_disease_specific[test_disease_code])}"
)
print(f"Complete disease-level input: {cir011_input_tokens:,} tokens")
print(f"Within 20k limit: {cir011_input_tokens <= 20_000}")

In [ ]:
# ============================================================
# Measure all Cardiology disease-level inputs
# ============================================================

cardiology_disease_inputs = {}

for disease_code, disease_name in cardiology_diseases.items():

    disease_input = build_cardiology_disease_input(
        disease_code=disease_code,
        disease_name=disease_name,
        patient_chunks=cardiology_results[disease_code],
        guideline_chunks=cardiology_kb_results_disease_specific[
            disease_code
        ]
    )

    token_count = len(
        tokenizer.encode(
            disease_input,
            add_special_tokens=False
        )
    )

    cardiology_disease_inputs[disease_code] = {
        "disease_name": disease_name,
        "input_text": disease_input,
        "token_count": token_count
    }

    print(
        f"{disease_code} | {disease_name}: "
        f"{token_count:,} tokens | "
        f"within 20k: {token_count <= 20_000}"
    )

In [ ]:
# ============================================================
# Run one disease-level Cardiology assessment: CIR011
# ============================================================

def run_cardiology_disease_assessment(
    disease_input,
    model,
    processor,
    max_new_tokens=1200
):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": disease_input
                }
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    input_length = inputs["input_ids"].shape[-1]

    print(
        f"Disease assessment input: "
        f"{input_length:,} tokens"
    )

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=4
        )

    generated_tokens = outputs[0][input_length:]

    assessment = processor.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return {
        "assessment": assessment,
        "input_token_count": input_length,
        "generated_token_count": len(generated_tokens),
        "hit_generation_limit": (
            len(generated_tokens) >= max_new_tokens
        )
    }


cir011_assessment_test = run_cardiology_disease_assessment(
    disease_input=cardiology_disease_inputs["CIR011"]["input_text"],
    model=model,
    processor=processor,
    max_new_tokens=1200
)

print(
    f"\nGenerated: "
    f"{cir011_assessment_test['generated_token_count']:,} tokens"
)
print(
    f"Hit generation limit: "
    f"{cir011_assessment_test['hit_generation_limit']}"
)

print("\n" + "=" * 80)
print("CIR011 DISEASE-LEVEL ASSESSMENT")
print("=" * 80)

print(cir011_assessment_test["assessment"])

In [ ]:
# ============================================================
# Run all 5 Cardiology disease-level assessments
# ============================================================

cardiology_disease_assessments = {}

for disease_code, disease_name in cardiology_diseases.items():

    print("\n" + "=" * 80)
    print(f"{disease_code} | {disease_name}")
    print("=" * 80)

    # Build disease-specific input using the finalized prompt
    disease_input = build_cardiology_disease_input(
        disease_code=disease_code,
        disease_name=disease_name,
        patient_chunks=cardiology_results[disease_code],
        guideline_chunks=cardiology_kb_results_disease_specific[
            disease_code
        ]
    )

    # Run MedGemma
    result = run_cardiology_disease_assessment(
        disease_input=disease_input,
        model=model,
        processor=processor,
        max_new_tokens=500
    )

    # Store result under the correct disease code
    cardiology_disease_assessments[disease_code] = {
        "disease_name": disease_name,
        "assessment": result["assessment"],
        "input_token_count": result["input_token_count"],
        "generated_token_count": result["generated_token_count"],
        "hit_generation_limit": result["hit_generation_limit"]
    }

    print(
        f"\nGenerated: "
        f"{result['generated_token_count']:,} tokens"
    )

    print(
        f"Hit generation limit: "
        f"{result['hit_generation_limit']}"
    )

    print("\n" + result["assessment"])

In [ ]:
# ============================================================
# Rerun truncated CIR008 assessment only
# ============================================================

disease_code = "CIR008"
disease_name = cardiology_diseases[disease_code]

cir008_input = build_cardiology_disease_input(
    disease_code=disease_code,
    disease_name=disease_name,
    patient_chunks=cardiology_results[disease_code],
    guideline_chunks=cardiology_kb_results_disease_specific[
        disease_code
    ]
)

cir008_result = run_cardiology_disease_assessment(
    disease_input=cir008_input,
    model=model,
    processor=processor,
    max_new_tokens=750
)

print(
    f"\nGenerated: "
    f"{cir008_result['generated_token_count']:,} tokens"
)

print(
    f"Hit generation limit: "
    f"{cir008_result['hit_generation_limit']}"
)

print("\n" + "=" * 80)
print("CIR008 RERUN")
print("=" * 80)

print(cir008_result["assessment"])

In [ ]:
# ============================================================
# Replace truncated CIR008 result with completed rerun
# ============================================================

cardiology_disease_assessments["CIR008"] = {
    "disease_name": cardiology_diseases["CIR008"],
    "assessment": cir008_result["assessment"],
    "input_token_count": cir008_result["input_token_count"],
    "generated_token_count": cir008_result["generated_token_count"],
    "hit_generation_limit": cir008_result["hit_generation_limit"]
}

print("CIR008 assessment updated.")

print(
    "Hit generation limit:",
    cardiology_disease_assessments["CIR008"][
        "hit_generation_limit"
    ]
)

print(
    "\nStored assessment:\n"
    + cardiology_disease_assessments["CIR008"]["assessment"]
)

In [ ]:
# ============================================================
# Final Cardiology synthesis prompt
# ============================================================

CARDIOLOGY_FINAL_SYNTHESIS_PROMPT = """
You are the final Cardiology specialist in a multi-specialist
clinical reasoning system.

You will receive five independent disease-specific assessments for
the same patient.

The five candidate diagnosis categories are:

- CIR017: Cardiac dysrhythmias
- CIR019: Heart failure
- CIR008: Hypertension with complications and secondary hypertension
- CIR011: Coronary atherosclerosis and other heart disease
- CIR007: Essential hypertension

Your task is to critically reconcile the five disease-specific
assessments and produce the final Cardiology prediction at the end
of the admission.

IMPORTANT:
The disease-specific assessments are intermediate model outputs.
They are NOT ground truth.

They may contain:
- overlapping evidence,
- irrelevant evidence,
- unsupported inferences,
- incorrect attribution of evidence to a candidate,
- conflicting conclusions,
- management suggestions derived from guidelines rather than
  documented patient management.

Therefore, do NOT automatically accept a disease simply because its
intermediate assessment says "Supported".

DIAGNOSIS RULES:
- Consider only the five predefined candidate categories.
- Select only diagnoses adequately supported by the evidence described
  in the disease-specific assessments.
- A diagnosis does not need to be the primary reason for admission.
- Chronic disease may be retained when the assessments contain clear
  evidence that it is an established and clinically relevant condition.
- Distinguish established disease from merely suspected disease.
- Do not use evidence for one disease as proof of another disease
  unless the relationship is explicitly supported.
- Reconcile overlap between CIR007 and CIR008 rather than automatically
  selecting both.
- Do not invent additional diagnoses.
- It is acceptable for none of the five diagnoses to be supported.

MANAGEMENT RULES:
- Produce Cardiology discharge/end-of-admission management relevant
  to the final supported diagnoses.
- Distinguish documented patient management from guideline-derived
  considerations.
- Do not state that a guideline recommendation was actually performed
  unless an intermediate assessment identifies it as documented
  patient management.
- Do not invent medications, procedures, investigations, referrals,
  or follow-up.
- Remove management recommendations that are unrelated to the final
  supported Cardiology diagnoses.
- Reconcile duplicate or conflicting management items.

Return exactly:

SUPPORTED CARDIOLOGY DIAGNOSES:
- <code>: <diagnosis> — <brief evidence basis>

or

- None supported

DOCUMENTED CARDIOLOGY MANAGEMENT:
- <documented management relevant to the supported diagnoses>

or

- None identified

GUIDELINE-SUPPORTED CARDIOLOGY CONSIDERATIONS:
- <relevant consideration grounded in the supplied disease-specific
  assessments>

or

- None identified

Do not assess diagnoses outside the five predefined Cardiology
candidate categories.
""".strip()

In [ ]:
# ============================================================
# Build and measure final Cardiology synthesis input
# ============================================================

assessment_blocks = []

for disease_code, disease_name in cardiology_diseases.items():

    assessment_text = cardiology_disease_assessments[
        disease_code
    ]["assessment"]

    assessment_blocks.append(
        f"CANDIDATE: {disease_code} | {disease_name}\n\n"
        f"{assessment_text}"
    )


combined_cardiology_assessments = (
    "\n\n" + "=" * 80 + "\n\n"
).join(assessment_blocks)


cardiology_final_input = (
    CARDIOLOGY_FINAL_SYNTHESIS_PROMPT
    + "\n\n"
    + "=" * 80
    + "\n\nDISEASE-SPECIFIC ASSESSMENTS:\n\n"
    + combined_cardiology_assessments
)


cardiology_final_input_tokens = len(
    tokenizer.encode(
        cardiology_final_input,
        add_special_tokens=False
    )
)


print(
    f"Number of disease assessments: "
    f"{len(cardiology_disease_assessments)}"
)

print(
    f"Final Cardiology synthesis input: "
    f"{cardiology_final_input_tokens:,} tokens"
)

print(
    f"Within 20k limit: "
    f"{cardiology_final_input_tokens <= 20_000}"
)

In [ ]:
# ============================================================
# Run final Cardiology specialist synthesis
# ============================================================

def run_cardiology_final_synthesis(
    synthesis_input,
    model,
    processor,
    max_new_tokens=800
):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": synthesis_input
                }
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    input_length = inputs["input_ids"].shape[-1]

    print(
        f"Final Cardiology input: "
        f"{input_length:,} tokens"
    )

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=4
        )

    generated_tokens = outputs[0][input_length:]

    final_output = processor.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return {
        "final_output": final_output,
        "input_token_count": input_length,
        "generated_token_count": len(generated_tokens),
        "hit_generation_limit": (
            len(generated_tokens) >= max_new_tokens
        )
    }


cardiology_final_result = run_cardiology_final_synthesis(
    synthesis_input=cardiology_final_input,
    model=model,
    processor=processor,
    max_new_tokens=800
)


print(
    f"\nGenerated: "
    f"{cardiology_final_result['generated_token_count']:,} tokens"
)

print(
    f"Hit generation limit: "
    f"{cardiology_final_result['hit_generation_limit']}"
)

print("\n" + "=" * 80)
print("FINAL CARDIOLOGY SPECIALIST OUTPUT")
print("=" * 80)

print(cardiology_final_result["final_output"])